# Scenario 4 — Train Without Midday/Midnight, Predict Both

**Midday/midnight defined via standard local time (fixed UTC offsets — DST not applied).**

| Window | Standard LT hours |
|--------|------------------|
| Midday | 11, 12, 13 |
| Midnight | 23, 0, 1 |

The `Hour` column in both CSVs already holds **standard local time** (written by `prepare_datasets.py`), so no UTC conversion is needed here.

| Station | UTC Offset |
|---------|------------|
| CP | UTC−3 (BRT) |
| ElginAB | UTC−6 (CST) |
| Jicamarca | UTC−5 (PET) |
| MilstonHill | UTC−5 (EST) |
| Ramey | UTC−4 (AST) |

**Training data:** `ann_evals/data/master_no_midday_midnight.csv` (Hour = std LT)  
**Evaluation:** `ann_evals/data/midday_midnight_only.csv` (Hour = std LT)  
**Outputs:** `ann_no_midday_midnight_model.pt`, loss curve, 3 scatter plots (combined + midday-only + midnight-only), predictions CSV

> Run `prepare_datasets.py` (or `prepare_datasets.ipynb`) first.

## 1. Imports and Path Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import warnings
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings('ignore')
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
import os
from pathlib import Path

_cwd = Path(os.getcwd())
if (_cwd / 'FINAL_MASTER.csv').exists():
    project_root = _cwd
elif (_cwd.parent.parent / 'FINAL_MASTER.csv').exists():
    project_root = _cwd.parent.parent
else:
    raise FileNotFoundError('Cannot find FINAL_MASTER.csv — run from project root or ann_evals/notebooks/')

ann_evals  = project_root / 'ann_evals'
data_dir   = ann_evals / 'data'
output_dir = ann_evals / 'outputs' / 'scenario4_no_midday_midnight'
output_dir.mkdir(parents=True, exist_ok=True)

TRAIN_CSV = data_dir / 'master_no_midday_midnight.csv'
EVAL_CSV  = data_dir / 'midday_midnight_only.csv'

RANDOM_STATE   = 42
FEATURES       = ['DayOfYear', 'Hour', 'Longitude', 'Latitude', 'F10.7']
TARGET         = 'foF2'
BATCH_SIZE     = 128
LR             = 1e-3
MAX_EPOCHS     = 200
PATIENCE       = 12
DEVICE         = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MIDDAY_HOURS   = {11, 12, 13}
MIDNIGHT_HOURS = {23, 0, 1}

STATIONS       = ['CP', 'ElginAB', 'Jicamarca', 'MilstonHill', 'Ramey']
STATION_COLOR  = dict(zip(STATIONS, ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']))
STATION_MARKER = dict(zip(STATIONS, ['o', 's', '^', 'D', 'v']))

print(f'Output dir : {output_dir}')
print(f'Device     : {DEVICE}')
print(f'Train CSV  : {TRAIN_CSV.exists()}')
print(f'Eval CSV   : {EVAL_CSV.exists()}')
print('Hour column: standard local time (no DST)')

## 2. Load Data

The `Hour` column is already standard local time — no UTC conversion needed. Midday/midnight periods are identified directly from the Hour values.

In [ ]:
df_train = pd.read_csv(TRAIN_CSV)
df_eval  = pd.read_csv(EVAL_CSV)
print(f'Training : {len(df_train):,} rows  |  Hour range: {df_train["Hour"].min()}–{df_train["Hour"].max()} (std LT)')
print(f'Eval     : {len(df_eval):,} rows  |  Hour range: {df_eval["Hour"].min()}–{df_eval["Hour"].max()} (std LT)')

# Derive midday/midnight masks directly from local-time Hour column
mask_midday   = df_eval['Hour'].isin(MIDDAY_HOURS).values
mask_midnight = df_eval['Hour'].isin(MIDNIGHT_HOURS).values
print(f'  Midday rows   (LT {{11,12,13}}): {mask_midday.sum():,}')
print(f'  Midnight rows (LT {{23,0,1}}) : {mask_midnight.sum():,}')

X_all  = df_train[FEATURES].values.astype(np.float32)
y_all  = df_train[TARGET].values.astype(np.float32).reshape(-1, 1)

X_eval = df_eval[FEATURES].values.astype(np.float32)
y_eval = df_eval[TARGET].values.astype(np.float32)
s_eval = df_eval['Station'].values

X_train, X_val, y_train, y_val = train_test_split(
    X_all, y_all, test_size=0.15/0.85, random_state=RANDOM_STATE)

train_mean = X_train.mean(axis=0)
train_std  = np.where(X_train.std(axis=0) == 0, 1.0, X_train.std(axis=0))
X_train_n  = (X_train - train_mean) / train_std
X_val_n    = (X_val   - train_mean) / train_std
X_eval_n   = (X_eval  - train_mean) / train_std
print(f'Train: {len(X_train):,}  Val: {len(X_val):,}')

## 3. Model and Training

In [ ]:
class ANNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(5,   16), nn.ReLU(),
            nn.Linear(16,  32), nn.ReLU(),
            nn.Linear(32,  64), nn.ReLU(),
            nn.Linear(64, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64,  32), nn.ReLU(),
            nn.Linear(32,  16), nn.ReLU(),
            nn.Linear(16,   1),
        )
    def forward(self, x):
        return self.net(x)

model     = ANNModel().to(DEVICE)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

def to_t(a): return torch.from_numpy(a).to(DEVICE)

loader = DataLoader(TensorDataset(to_t(X_train_n), to_t(y_train)),
                    batch_size=BATCH_SIZE, shuffle=True)
X_va_t = to_t(X_val_n);  y_va_t = to_t(y_val)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
best_val_loss, best_epoch, patience_count, best_state = float('inf'), 0, 0, None
train_losses, val_losses = [], []

for epoch in range(MAX_EPOCHS):
    model.train()
    epoch_loss = 0.0
    for Xb, yb in loader:
        optimizer.zero_grad()
        loss = criterion(model(Xb), yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * Xb.size(0)
    epoch_loss /= len(X_train)
    train_losses.append(epoch_loss)

    model.eval()
    with torch.no_grad():
        val_loss = criterion(model(X_va_t), y_va_t).item()
    val_losses.append(val_loss)

    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1:3d}/{MAX_EPOCHS}  train={epoch_loss:.5f}  val={val_loss:.5f}')

    if val_loss < best_val_loss:
        best_val_loss, best_epoch, patience_count = val_loss, epoch, 0
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        patience_count += 1
        if patience_count >= PATIENCE:
            print(f'Early stop @ epoch {epoch+1}')
            break

model.load_state_dict(best_state)
print(f'Best epoch: {best_epoch+1}  val MSE={best_val_loss:.5f}')

## 4. Save Checkpoint and Loss Curve

In [ ]:
torch.save({
    'model_state_dict': model.state_dict(),
    'train_mean':       train_mean,
    'train_std':        train_std,
    'features':         FEATURES,
    'epoch':            best_epoch,
}, output_dir / 'ann_no_midday_midnight_model.pt')

plt.figure(figsize=(9, 5))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses,   label='Val Loss')
plt.axvline(best_epoch, color='r', linestyle='--', label=f'Best epoch ({best_epoch+1})')
plt.xlabel('Epoch'); plt.ylabel('MSE Loss')
plt.title('Scenario 4 — No Midday/Midnight: Training & Validation Loss')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(output_dir / 'ann_no_midday_midnight_loss_curve.png', dpi=150)
plt.show()
print('Checkpoint and loss curve saved.')

## 5. Evaluate — Metrics

In [ ]:
model.eval()
with torch.no_grad():
    y_pred = model(to_t(X_eval_n)).cpu().numpy().flatten()

def print_metrics(label, yt, yp, st):
    mse = mean_squared_error(yt, yp)
    r2  = r2_score(yt, yp)
    print(f'{label}  MSE={mse:.4f}  RMSE={np.sqrt(mse):.4f}  '
          f'MAE={mean_absolute_error(yt, yp):.4f}  R2={r2:.4f}  N={len(yt):,}')
    for s in STATIONS:
        m = st == s
        if not m.any(): continue
        rmse_s = float(np.sqrt(mean_squared_error(yt[m], yp[m])))
        r2_s   = float(r2_score(yt[m], yp[m]))
        bias_s = float(np.mean(yp[m] - yt[m]))
        print(f'  {s:<14} RMSE={rmse_s:.4f}  R2={r2_s:.4f}  Bias={bias_s:+.4f}  N={m.sum()}')

print_metrics('ALL MIDDAY+MIDNIGHT', y_eval, y_pred, s_eval)
print()
print_metrics('MIDDAY ONLY',   y_eval[mask_midday],   y_pred[mask_midday],   s_eval[mask_midday])
print()
print_metrics('MIDNIGHT ONLY', y_eval[mask_midnight], y_pred[mask_midnight], s_eval[mask_midnight])

## 6. Combined Scatter Plot (Midday + Midnight)

Filled markers = midday hours (LT 11–13), hollow markers = midnight hours (LT 23, 0, 1). Both periods share the same station colour.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

for s in STATIONS:
    mask = s_eval == s
    if not mask.any():
        continue
    if mask_midday[mask].any():
        ax.scatter(y_eval[mask & mask_midday], y_pred[mask & mask_midday],
                   s=8, alpha=0.5, color=STATION_COLOR[s],
                   marker=STATION_MARKER[s], label=f'{s} midday')
    if mask_midnight[mask].any():
        ax.scatter(y_eval[mask & mask_midnight], y_pred[mask & mask_midnight],
                   s=8, alpha=0.5, color=STATION_COLOR[s],
                   marker=STATION_MARKER[s], facecolors='none',
                   edgecolors=STATION_COLOR[s], linewidths=0.8,
                   label=f'{s} midnight')

lims = [min(y_eval.min(), y_pred.min()) - 0.3, max(y_eval.max(), y_pred.max()) + 0.3]
ax.plot(lims, lims, 'k--', lw=1)
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel('Observed foF2 (MHz)')
ax.set_ylabel('Predicted foF2 (MHz)')
ax.set_title('Scenario 4 — Midday & Midnight Predictions\n'
             '(filled = midday, hollow = midnight)', fontsize=10)
ax.legend(fontsize=6, markerscale=1.8, ncol=2)

rmse = float(np.sqrt(mean_squared_error(y_eval, y_pred)))
mae  = float(mean_absolute_error(y_eval, y_pred))
r2   = float(r2_score(y_eval, y_pred))
bias = float(np.mean(y_pred - y_eval))
txt  = (f'RMSE={rmse:.3f}  MAE={mae:.3f}\n'
        f'R2={r2:.4f}   Bias={bias:+.3f}\nN={len(y_eval):,}')
ax.text(0.03, 0.97, txt, transform=ax.transAxes, va='top', fontsize=8,
        bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.8))

plt.tight_layout()
plt.savefig(output_dir / 'ann_no_midday_midnight_scatter.png', dpi=150)
plt.show()
print('Scatter plot saved.')

## 7. Separate Scatter Plots (Midday Only / Midnight Only)

In [ ]:
def scatter_plot(y_obs, y_pred_arr, st, title, fname):
    fig, ax = plt.subplots(figsize=(6, 5))
    for s in STATIONS:
        m = st == s
        if not m.any():
            continue
        ax.scatter(y_obs[m], y_pred_arr[m], s=8, alpha=0.5,
                   color=STATION_COLOR[s], marker=STATION_MARKER[s], label=s)
    lims = [min(y_obs.min(), y_pred_arr.min()) - 0.3,
            max(y_obs.max(), y_pred_arr.max()) + 0.3]
    ax.plot(lims, lims, 'k--', lw=1)
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel('Observed foF2 (MHz)')
    ax.set_ylabel('Predicted foF2 (MHz)')
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=7, markerscale=1.5)
    rmse = float(np.sqrt(mean_squared_error(y_obs, y_pred_arr)))
    mae  = float(mean_absolute_error(y_obs, y_pred_arr))
    r2   = float(r2_score(y_obs, y_pred_arr))
    bias = float(np.mean(y_pred_arr - y_obs))
    txt  = (f'RMSE={rmse:.3f}  MAE={mae:.3f}\n'
            f'R2={r2:.4f}   Bias={bias:+.3f}\nN={len(y_obs):,}')
    ax.text(0.03, 0.97, txt, transform=ax.transAxes, va='top', fontsize=8,
            bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.8))
    plt.tight_layout()
    plt.savefig(output_dir / fname, dpi=150)
    plt.show()
    print(f'Saved: {fname}')

scatter_plot(y_eval[mask_midday], y_pred[mask_midday], s_eval[mask_midday],
             'Scenario 4 — Midday Predictions (std LT 11–13)',
             'ann_no_midday_scatter.png')

scatter_plot(y_eval[mask_midnight], y_pred[mask_midnight], s_eval[mask_midnight],
             'Scenario 4 — Midnight Predictions (std LT 23, 0, 1)',
             'ann_no_midnight_scatter.png')

## 7. Save Predictions

In [ ]:
period_col = ['midday' if h in MIDDAY_HOURS else 'midnight'
              for h in df_eval['Hour'].values]

out_df = pd.DataFrame({
    'Station':   s_eval,
    'DayOfYear': df_eval['DayOfYear'].values,
    'Hour':      df_eval['Hour'].values,   # standard local time
    'period':    period_col,
    'foF2_obs':  y_eval,
    'foF2_pred': y_pred,
})
out_df.to_csv(output_dir / 'predictions_ann_no_midday_midnight.csv', index=False)
print(f'Predictions saved  ({len(out_df):,} rows)')
out_df.head(10)